I am training a model on heart disease dataset

In [ ]:
#writing requirements file for python package version needed
requirements = """
numpy>=1.24.0
matplotlib>=3.7.0
seaborn>=0.12.0
plotly>5.15.0
ydata-profiling>=4.5.0
scikit-learn>=1.2.0
###dataset package
ucimlrepo==0.0.7
joblib>=1.2.0
"""
with open("requirements.txt","w") as f:
    f.write(requirements)

: 

In [2]:
#installing the packages from requirements file
%pip install -r requirements.txt

In [3]:
from ucimlrepo import list_available_datasets

#listing the datasets available in the ucimlrepo package
list_available_datasets()

In [4]:
#retrieve the dataset from the ucimlrepo package
from ucimlrepo import fetch_ucirepo as fetch_dataset
import pandas as pd

heart_disease = fetch_dataset(name="Heart Disease")
heart_disease_df = pd.concat([heart_disease.data['features'], heart_disease.data['targets']], axis=1)
#save the dataset as a csv file
heart_disease_df.to_csv("heart_disease.csv", index=False)


In [5]:
#loading the dataset and inspecting the dataset

dataset = pd.read_csv("heart_disease.csv")
dataset.shape

In [6]:
from ydata_profiling import ProfileReport

#performing exploratory data analysis using the ydata_profiling package and
#  saving the report as an html file
profile = ProfileReport(dataset, title="Heart Disease EDA Report", explorative=True)
profile.to_file("heart_disease_report.html")

In [7]:
dataset.info()

In [8]:
dataset.head()

In [9]:
#rename the target column to "target"
dataset["target"] = dataset['num']
dataset.drop(columns=["num"], inplace=True)
dataset.head()

In [10]:
#change the target column to binary values since the target column has values from 0 to 4 where 0 
# means no heart disease and 1,2,3,4 means heart disease
dataset["target"] = dataset['target'].apply(lambda s: 1 if s>0 else 0)
dataset.head()

In [11]:
dataset.duplicated().sum()

In [12]:
dataset.isnull().sum()

In [13]:
#checking the dataset for target data imbalance
dataset["target"].value_counts()/len(dataset)

In [14]:
#plotting the target variable distribution using seaborn
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(x="target", data=dataset)
plt.title("Distribution of Target Variable")
plt.xlabel("Target")
plt.ylabel("Count")
plt.show()

In [15]:
#checking the outliers in the dataset using boxplot
numeric_columns = dataset.drop(columns=["target","sex","cp","fbs","restecg","exang","slope","ca","thal"])
for column in numeric_columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=dataset[column])
    plt.title(f"Boxplot of {column}")
    plt.show()

Because oldpeak, chol, trestbps have outliers i am using robustscaler since i dont want to remove it
Now i need to split the dataset then impute the missing value then scale it to remove the outlier

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from ydata_profiling import ProfileReport
import pandas as pd
from sklearn.model_selection import train_test_split

# prepare X and y
X = dataset.drop(columns=['target'])
y = dataset['target']

numerical_col = ["age", "trestbps", "chol", "thalach", "oldpeak"]
category_col = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]  # removed 'target'

numerical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])
category_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

preproc = ColumnTransformer([
    ('num', numerical_pipe, numerical_col),
    ('cat', category_pipe, category_col)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_processed = preproc.fit_transform(X_train)
X_test_processed = preproc.transform(X_test)

# get feature names (fallback to manual list if not available)
try:
    feature_names = preproc.get_feature_names_out()
except Exception:
    feature_names = numerical_col + category_col

X_train_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_train_df['target'] = y_train.values

profile = ProfileReport(X_train_df, title="Train split EDA", explorative=True)
profile.to_file("train_split_profile.html")
X_test_processed = preproc.transform(X_test)



In [20]:
from google.colab import files
files.download('/content/train_split_profile.html')


In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(),
    "XGBoost": XGBClassifier(),
    "Support Vector Machine": SVC()
}

for name, model in models.items():
    pipe = Pipeline([('preproc', preproc), ('clf', model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='f1')
    print(f"{name} F1 Score: {scores.mean():.4f} (+/- {scores.std():.4f})")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score,precision_score,recall_score,confusion_matrix,roc_auc_score
import joblib

pipe = Pipeline([('preproc', preproc),('clf', LogisticRegression())])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe.named_steps['clf'], "predict_proba") else None  

joblib.dump(pipe, 'heart_disease_model.joblib')

print('F1', f1_score(y_test, y_pred))
print('Precision', precision_score(y_test, y_pred))
print('Recall', recall_score(y_test, y_pred))
if y_proba is not None:
    print('ROC AUC', roc_auc_score(y_test, y_proba))
print('Confusion matrix:\\n', confusion_matrix(y_test, y_pred))